In [1]:
using Pkg
Pkg.activate(".")
using MLStyle

  Activating project at `~/Documents/GitHub/Julia_Tutorials/CategoryTheory`


## Comonads

A comonad is a triple $(W,\epsilon,\delta)$, where
$$W:\mathcal C \to \mathcal C$$
$$\epsilon: W \Rightarrow Id_\mathcal C$$
$$\delta: W \Rightarrow W \circ W$$

The counit ($\epsilon$) is commonly called `extract`, since $\epsilon_A: WA \to A$.

The codiagonal ($\delta$) is commonly called `duplicate`, since $\delta_A: WA \to W WA$.

Lastly, we have `extend`, which is just another way of using these natural transformations.
More specifically, $\text{extend}:(WA\to B)\times WA \to WB$. Thus, given a function $f:WA \to B$, and a value
 $x$ of $WA$, the `extend` simply computes 
$$
\text{extend}(f,x) := Wf \circ \delta
$$
Note that, $Wf: WWA\to WB$, thus $Wf \circ \delta_A: WA \to W B$.


According to [this](https://github.com/Moelf/functional-programming-jargon.jl?tab=readme-ov-file#equational-reasoning]),
it is an object that has extract and extend functions. Ref is a natural condidate:

In [2]:
fmap(f::Function,x::Ref) = Ref(f(x[]))

extract(x::Ref) = x[]
extend(f,x::T) where T<:Ref = T(f(x[]))
duplicate(wa::Ref) = Ref(wa)

extract(Ref(10))

extend(x->3,Ref(1.0))

ext(f::Function,x::Ref) = fmap(f,duplicate(x))


ext(x->3,Ref(1.0))[] == extend(x->3,Ref(1.0))[]

true

### Non-empty Lists

Let $F_A X = 1 + A \times X$, the list of A functor is the fixed point 
$$
\mu L_A X=  1 + A \times \mu L_A X
$$

Another example of comonad is the non-empty list:
$$
NL_A X = A + \mu L_A
$$

In [42]:
[1,23][begin+1:end]

1-element Vector{Int64}:
 23

In [55]:
struct NL{A}
    head::A
    tail::Vector{A}
end
NL(a::A) where A = NL(a,A[])

ϵ(nl::NL) = nl.head

δ(tail::Vector) = begin
    if tail == []
        return []
    end
    vcat(NL(tail[1], tail[2:end]), lst2nlst(tail[2:end]))
end
δ(nl::NL) = NL(nl,δ(nl.tail))

nl = NL(0,[1,2,3,4])

δ(nl)

NL{Any}(NL{Int64}(0, [1, 2, 3, 4]), Any[NL{Int64}(1, [2, 3, 4]), NL{Int64}(2, [3, 4]), NL{Int64}(3, [4]), NL{Int64}(4, Int64[])])

In [57]:
nl = NL(0,[])
δ(nl)

NL{Any}(NL{Any}(0, Any[]), Any[])